In [ ]:
import pandas as pd
from tabulate import tabulate

# Create each data file with its expected columns if it does not already exist.
try:
    login_df = pd.read_csv("login.csv")
except:
    login_df = pd.DataFrame({"username": ["Admin"], "password": ["Pass"]})
    login_df.to_csv("login.csv", index=False)

try:
    stock_df = pd.read_csv("stock.csv")
except:
    stock_df = pd.DataFrame(columns=["icode", "iname", "itype", "qty", "price"])
    stock_df.to_csv("stock.csv", index=False)

try:
    purchase_df = pd.read_csv("purchase.csv")
except:
    purchase_df = pd.DataFrame(columns=["bdate", "name", "icode", "qty", "amount"])
    purchase_df.to_csv("purchase.csv", index=False)


# Read the current stock records from the CSV file.
def read_stock():
    return pd.read_csv("stock.csv")

# Save stock changes so they persist between program runs.
def write_stock(df):
    df.to_csv("stock.csv", index=False)

# Read the admin credentials from the CSV file.
def read_login():
    return pd.read_csv("login.csv")

# Save updated admin credentials.
def write_login(df):
    df.to_csv("login.csv", index=False)

# Read the saved customer purchase records.
def read_purchase():
    return pd.read_csv("purchase.csv")

# Save purchase history to the CSV file.
def write_purchase(df):
    df.to_csv("purchase.csv", index=False)

# Display a DataFrame as a formatted console table.
def show_table(df):
    print(tabulate(df, headers="keys", tablefmt="grid", showindex=False))

# Keep showing the main menu until the user chooses to exit.
while True:
    print("\n-----------------------------------")
    print(" Welcome to Gamer's Stop ")
    print("-----------------------------------")
    print("1. Admin\n2. Customer\n3. Exit\n")

    ch = int(input("Enter your choice: "))

    # Authenticate the admin before displaying inventory management options.
    if ch == 1:
        passwd = input("Enter the password: ")
        login_df = read_login()

        if passwd == login_df.loc[0, "password"]:
            # Keep the admin menu open until the admin logs out.
            while True:
                print("\n==== Welcome Admin ====")
                print(
                    "1. Add New Item\n"
                    "2. Delete Item\n"
                    "3. Update Price\n"
                    "4. Update Quantity\n"
                    "5. Display All Items\n"
                    "6. Change Password\n"
                    "7. Log Out\n"
                )

                ch = int(input("Enter your choice: "))
                stock_df = read_stock()

                # Add a new game or gaming device to the inventory.
                if ch == 1:
                    icode = int(input("Enter Item Code: "))
                    iname = input("Enter Item Name: ")
                    itype = input("Game or Gaming Device?: ")
                    qty = int(input("Enter Quantity: "))
                    price = float(input("Enter Price: "))

                    stock_df.loc[len(stock_df)] = [icode, iname, itype, qty, price]
                    write_stock(stock_df)
                    print("\nItem added successfully")

                # Remove the selected item code from the inventory.
                elif ch == 2:
                    icode = int(input("Enter Item Code to delete: "))
                    stock_df = stock_df[stock_df["icode"] != icode]
                    write_stock(stock_df)
                    print("\nItem deleted successfully")

                # Update the price for the selected item code.
                elif ch == 3:
                    icode = int(input("Enter Item Code: "))
                    new_price = float(input("Enter New Price: "))
                    stock_df.loc[stock_df["icode"] == icode, "price"] = new_price
                    write_stock(stock_df)
                    print("\nPrice updated successfully")

                # Update the available quantity for the selected item code.
                elif ch == 4:
                    icode = int(input("Enter Item Code: "))
                    new_qty = int(input("Enter New Quantity: "))
                    stock_df.loc[stock_df["icode"] == icode, "qty"] = new_qty
                    write_stock(stock_df)
                    print("\nQuantity updated successfully")

                # Show the full inventory, or a message if it is empty.
                elif ch == 5:
                    stock_df = read_stock()
                    print("\n====== Stock Available ======\n")
                    if stock_df.empty:
                        print("No items available")
                    else:
                        show_table(stock_df)

                # Verify the current password before saving a replacement.
                elif ch == 6:
                    old_pw = input("Enter old password: ")
                    if old_pw == login_df.loc[0, "password"]:
                        new_pw = input("Enter new password: ")
                        login_df.loc[0, "password"] = new_pw
                        write_login(login_df)
                        print("\nPassword changed successfully")
                    else:
                        print("\nIncorrect password")

                # Return to the main menu.
                elif ch == 7:
                    break
                else:
                    print("\nInvalid option")

        else:
            print("\nWrong password")

    # Let customers browse stock and purchase an available item.
    elif ch == 2:
        stock_df = read_stock()

        if stock_df.empty:
            print("\nNo items available")
            continue

        print("\n====== Available Items ======\n")
        show_table(stock_df[["icode", "iname", "qty", "price"]])

        print("\n1. Buy Item\n2. Cancel Purchase\n")
        ch = int(input("Enter your choice: "))

        if ch == 1:
            name = input("Enter your name: ")
            icode = int(input("Enter Item Code: "))
            qty = int(input("Enter Quantity: "))

            stock_df = read_stock()
            item = stock_df[stock_df["icode"] == icode]

            if item.empty:
                print("\nInvalid Item Code")
                continue

            item = item.iloc[0]

            if qty > item["qty"]:
                print("\nInsufficient stock")
                continue

            # Calculate the purchase total and update or remove the stock row.
            amount = qty * item["price"]
            remaining_qty = item["qty"] - qty

            if remaining_qty == 0:
                stock_df = stock_df[stock_df["icode"] != icode]
            else:
                stock_df.loc[stock_df["icode"] == icode, "qty"] = remaining_qty

            write_stock(stock_df)

            # Append the completed purchase to the permanent purchase history.
            purchase_df = read_purchase()
            purchase_df.loc[len(purchase_df)] = [
                pd.Timestamp.now().date(),
                name,
                icode,
                qty,
                amount
            ]
            write_purchase(purchase_df)

            print(f"\nAmount to be paid: ₹{amount}")

        elif ch == 2:
            continue
        else:
            print("\nInvalid option")

    # End the application when the user selects Exit.
    elif ch == 3:
        print("\nThank you for visiting Gamer's Stop!")
        break
    else:
        print("\nInvalid option")
